# VoiceHire-NLP Final Notebook

This notebook documents the multi-turn mock interview system and maps it to the required NLP course concepts.


## Required Interview Pipeline

1. Interviewer generates a question.
2. Browser TTS speaks the question in the Streamlit app.
3. Student records a voice answer.
4. Whisper transcribes the answer.
5. The evaluator scores relevance, structure, clarity, confidence, keyword coverage, and fluency.
6. The next question visibly references the previous answer.
7. The process repeats for 10 questions.
8. A final feedback report is generated.


## Lab Mapping

- Bag of Words: keyword coverage score
- TF-IDF: important concept extraction
- N-Gram model: fluency and repetition analysis
- POS tagging: sentence structure analysis
- NER: technologies, companies, and skill entities
- Text classification: good vs needs improvement label
- Evaluation metrics: multi-axis scoring and final report


In [ ]:
from src.audio_analysis import analyze_transcript_delivery
from src.evaluator import evaluate_answer
from src.interviewer import generate_initial_question, generate_follow_up
from src.memory_manager import InterviewMemory
from src.report_generator import build_report


In [ ]:
role = 'Python Developer'
memory = InterviewMemory(role=role)
question = generate_initial_question(role)

sample_answers = [
    'Python is useful because it has simple syntax, classes, APIs, and database libraries.',
    'I use classes to organize code and inheritance when shared behavior is needed.',
    'In a project I built an API, tested endpoints, and fixed database errors.',
    'I evaluate solutions using unit tests, logs, and expected outputs.',
    'A tradeoff is choosing simple code first instead of overengineering.',
    'For debugging I reproduce the issue, inspect logs, and isolate the failing function.',
    'For non technical people I explain the user impact and avoid unnecessary jargon.',
    'Important tools include Python, GitHub, SQL, Flask, and testing frameworks.',
    'With incomplete requirements I ask questions, define assumptions, and build small prototypes.',
    'I am a good fit because I can write clear Python code and explain my decisions.'
]

for turn_number, answer in enumerate(sample_answers, start=1):
    delivery = analyze_transcript_delivery(answer, duration_seconds=45)
    evaluation = evaluate_answer(question, answer, role, memory.turns, delivery)
    memory.add_turn(
        question=question,
        answer=answer,
        evaluation=evaluation,
        audio_analysis={
            'words_per_minute': delivery.words_per_minute,
            'filler_words': delivery.filler_count,
            'pause_count': delivery.estimated_pause_count,
        },
    )
    if turn_number < 10:
        question = generate_follow_up(role, question, answer, evaluation, turn_number + 1)

len(memory.turns), memory.turns[-1].evaluation.total_score


In [ ]:
for turn in memory.turns:
    print(f'Q{turn.turn_number}: {turn.question}')
    print(f'Answer: {turn.answer}')
    print(f'Score: {turn.evaluation.total_score}/60')
    print('-' * 80)


In [ ]:
report = build_report(memory)
print(report.as_markdown())


## Failure Analysis

- Subjective scoring is difficult because technical keywords can make a weak answer look stronger than it is.
- Whisper transcription errors can change important words and reduce relevance or keyword coverage.
- Confidence scoring from pace and filler words is only an approximation of human communication quality.
